# 200 · Algorithms & complexity — procedural layer

**Mnemonic:** 2 = binary — compare two things, split in two (binary search, O(n²) pairwise comparisons).

**Codes covered:** 205, 211, 226, 230, 242, 253, 266, 270, 283, 290.

**Method:** for every code cell — read it, **PREDICT the output**, run it, compare. The gap between your prediction and reality is the lesson. Exactly one cell in this notebook is *supposed* to raise (242); it is marked `# INTENDED ERROR`.

## 205 · O(n²) quadratic time

Runtime grows with the square of n — typical of nested loops comparing all pairs; double n, quadruple time.

*A party where every guest must clink glasses with every other guest: 10 guests, a quick toast; 1000 guests, shattered glass till dawn.*

**Watch:** we count operations, never wall-clock. Predict both counters and their ratio before running.

In [1]:
def count_pair_ops(n):
    ops = 0
    for i in range(n):
        for j in range(n):
            ops += 1          # one "clink" per pair (i, j)
    return ops

small = count_pair_ops(200)
big = count_pair_ops(400)
print("n=200 ->", small, "operations")
print("n=400 ->", big, "operations")
print("ratio :", big / small, "  (double n -> ~4x the work)")

n=200 -> 40000 operations
n=400 -> 160000 operations
ratio : 4.0   (double n -> ~4x the work)


## 211 · Binary search

On sorted data, compare the middle element and discard the half that cannot contain the target — O(log n) halvings.

*Dictionary word hunt: flip open dead center — "M... my word starts with T" — rip away the left half and repeat. Ten rips find any word.*

**Watch:** predict the step counter for a 1,000,000-element range. log2(1,000,000) ≈ 19.9 — how many halvings can it possibly take?

In [2]:
def binary_search(a, target):
    lo, hi = 0, len(a) - 1
    steps = 0
    while lo <= hi:
        steps += 1
        mid = (lo + hi) // 2
        if a[mid] < target:
            lo = mid + 1
        elif a[mid] > target:
            hi = mid - 1
        else:
            return mid, steps
    return -1, steps

haystack = range(1_000_000)          # sorted 0..999_999
idx, steps = binary_search(haystack, 742_617)
print("found at index", idx, "in", steps, "steps")
print("a linear scan would have taken", 742_618, "steps")

found at index 742617 in 20 steps
a linear scan would have taken 742618 steps


In [3]:
import bisect

idx = bisect.bisect_left(range(1_000_000), 742_617)
print("bisect.bisect_left ->", idx, " (same answer, stdlib one-liner)")

bisect.bisect_left -> 742617  (same answer, stdlib one-liner)


## 226 · Sort stability

A stable sort keeps equal keys in original relative order, so chained sorts build multi-key orderings; mergesort/Timsort are stable, quicksort/heapsort are not.

*A stable horse barn: two horses both named Bay trot out the gate exactly in the order they trotted in. The unstable barn next door shuffles the twins nightly.*

**Watch:** after the second sort (by city), predict the order of the Paris rows. Does the rank order from the first sort survive inside each city?

In [4]:
records = [
    ("Ada", "Paris", 3),
    ("Bo",  "Tokyo", 1),
    ("Cy",  "Paris", 1),
    ("Dee", "Tokyo", 2),
    ("Eli", "Paris", 2),
]
print("original:")
for r in records:
    print("  ", r)

records.sort(key=lambda r: r[2])   # first pass: by rank
records.sort(key=lambda r: r[1])   # second pass: by city (stable!)

print("after sort by rank, then by city:")
for r in records:
    print("  ", r)
print("within each city the rank order survives -> two stable sorts = one multi-key sort")

original:
   ('Ada', 'Paris', 3)
   ('Bo', 'Tokyo', 1)
   ('Cy', 'Paris', 1)
   ('Dee', 'Tokyo', 2)
   ('Eli', 'Paris', 2)
after sort by rank, then by city:
   ('Cy', 'Paris', 1)
   ('Eli', 'Paris', 2)
   ('Ada', 'Paris', 3)
   ('Bo', 'Tokyo', 1)
   ('Dee', 'Tokyo', 2)
within each city the rank order survives -> two stable sorts = one multi-key sort


## 230 · Counting sort

For integer keys in a small range k: tally occurrences into a count array, then walk the tallies in order — O(n + k), stable, zero comparisons.

*A mail room with 100 numbered pigeonholes: nobody compares letters — each drops straight into its numbered hole, then you walk the wall left to right scooping them out in order.*

**Watch:** there is no `<` between elements anywhere. Predict the sorted output, then find the comparison the code never makes.

In [5]:
data = [4, 1, 3, 4, 0, 2, 4, 1, 9, 3, 0, 5]

k = max(data)
cnt = [0] * (k + 1)
for x in data:
    cnt[x] += 1                       # tally into pigeonhole x
out = [v for v in range(k + 1) for _ in range(cnt[v])]

print("input :", data)
print("sorted:", out)
print("no element was ever compared to another -- only counted")

input : [4, 1, 3, 4, 0, 2, 4, 1, 9, 3, 0, 5]
sorted: [0, 0, 1, 1, 2, 3, 3, 4, 4, 4, 5, 9]
no element was ever compared to another -- only counted


## 242 · Stack depth and overflow

Every active call holds a stack frame; recursing deeper than the limit crashes (Python default ~1000) — DEPTH matters, not total call count.

*A waiter stacking one plate per unfinished order: linear recursion over a million items piles plates to the ceiling until the tower crashes down — RecursionError.*

**Watch:** the next cell is SUPPOSED to fail. Predict the exception name before running, then read the traceback's repeating frame. The follow-up cell prints the limit and shows the loop that shrugs it off.

In [6]:
# INTENDED ERROR — read the traceback
def countdown(n):
    return 0 if n == 0 else countdown(n - 1)

countdown(100_000)   # depth 100_000 >> the recursion limit

RecursionError: maximum recursion depth exceeded

In [7]:
import sys

print("recursion limit:", sys.getrecursionlimit())

def countdown_iter(n):
    while n:
        n -= 1               # same work, ZERO extra stack frames
    return 0

print("iterative countdown(100_000) ->", countdown_iter(100_000))
print("a loop keeps depth at 1 no matter how many iterations it runs")

recursion limit: 3000
iterative countdown(100_000) -> 0
a loop keeps depth at 1 no matter how many iterations it runs


## 253 · Memoization (top-down)

Wrap the natural recursion with a cache keyed by arguments: repeat calls return the stored answer; only reachable subproblems ever get computed.

*A student taping a MEMO to the fridge after every hard calculation: at the next midnight craving she reads the memo and skips the math. The fridge fills lazily, only with questions actually asked.*

**Watch:** same function body, same answer — predict BOTH call counts. Why can the cached version never exceed 25 calls for fib(24)?

In [8]:
import functools

naive_calls = 0
def fib_naive(n):
    global naive_calls
    naive_calls += 1
    return n if n < 2 else fib_naive(n - 1) + fib_naive(n - 2)

memo_calls = 0
@functools.lru_cache(maxsize=None)
def fib_memo(n):
    global memo_calls
    memo_calls += 1          # runs only on a cache MISS
    return n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)

print("fib_naive(24) =", fib_naive(24), "in", naive_calls, "calls")
print("fib_memo(24)  =", fib_memo(24), "in", memo_calls, "calls")
print("25 distinct subproblems (n = 0..24) -> at most 25 real calls")

fib_naive(24) = 46368 in 150049 calls
fib_memo(24)  = 46368 in 25 calls
25 distinct subproblems (n = 0..24) -> at most 25 real calls


## 266 · Greedy coin-change failure

Greedy "largest coin first" works for canonical systems (US coins) but fails in general: with coins {1, 3, 4}, greedy makes 6 as 4+1+1; optimal is 3+3.

*A proud cashier in the Kingdom of 1-3-4 pennies slaps down the big 4 for a 6-cent sale, then fumbles two more coins — while the customer holds up two shiny 3s and smirks.*

**Watch:** predict both coin lists before running. Where exactly does grabbing the 4 lock greedy out of the optimum?

In [9]:
coins, amount = [1, 3, 4], 6

def greedy(coins, amount):
    picked = []
    for c in sorted(coins, reverse=True):   # largest coin first
        while amount >= c:
            amount -= c
            picked.append(c)
    return picked

def dp_min_coins(coins, amount):
    best = [0] + [None] * amount            # best[a] = fewest coins for a
    take = [0] * (amount + 1)
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and best[a - c] is not None:
                if best[a] is None or best[a - c] + 1 < best[a]:
                    best[a], take[a] = best[a - c] + 1, c
    picked = []
    while amount:
        picked.append(take[amount])
        amount -= take[amount]
    return picked

g, d = greedy(coins, amount), dp_min_coins(coins, amount)
print("greedy :", " + ".join(map(str, g)), "->", len(g), "coins")
print("dp     :", " + ".join(map(str, d)), "->", len(d), "coins")
print("largest-first is a trap outside canonical coin systems")

greedy : 4 + 1 + 1 -> 3 coins
dp     : 3 + 3 -> 2 coins
largest-first is a trap outside canonical coin systems


## 270 · Breadth-first search

Explore level by level with a queue; each node is first reached along a fewest-edges path, giving shortest paths in unweighted graphs.

*A stone dropped in a pond: perfect rings ripple outward, soaking the distance-1 lilies, then the distance-2 — every lily is first touched by its shortest ripple.*

**Watch:** predict the visit order (level by level from A) and the path to G before running. What different order would a stack give instead of a queue?

In [10]:
from collections import deque

graph = {
    "A": ["B", "C"],
    "B": ["A", "D", "E"],
    "C": ["A", "F"],
    "D": ["B"],
    "E": ["B", "F"],
    "F": ["C", "E", "G"],
    "G": ["F"],
}

start, target = "A", "G"
q = deque([start])
parent = {start: None}          # doubles as the visited set
order = []
while q:
    node = q.popleft()          # queue -> ripple outward by level
    order.append(node)
    for nb in graph[node]:
        if nb not in parent:
            parent[nb] = node
            q.append(nb)

path, node = [], target
while node is not None:
    path.append(node)
    node = parent[node]
path.reverse()

print("visit order   :", " ".join(order))
print("shortest A->G :", " -> ".join(path), f"({len(path) - 1} edges)")
print("first arrival = fewest edges, guaranteed by the queue")

visit order   : A B C D E F G
shortest A->G : A -> C -> F -> G (3 edges)
first arrival = fewest edges, guaranteed by the queue


## 283 · Fisher-Yates shuffle

Uniform in-place shuffle: walk i from the end, swapping a[i] with a[j] for random j ≤ i; every permutation equally likely, O(n).

*Fisher nets one random fish from the shrinking barrel and freezes it against the barrel's back wall; the frozen row grows, the swimming pool shrinks — an honest shuffle, fish by fish.*

**Watch:** the naive version draws j from the WHOLE list every step — 3³ = 27 outcomes cannot split evenly over 3! = 6 permutations. Exact math says B lands at position 0 in 10/27 of naive runs but only at 8/27 at position 1 (uniform would be 9/27 everywhere). Predict which frequency table looks flat (~1000 each) and which does not.

In [11]:
import random
from collections import Counter

def naive_shuffle(a, rng):
    for i in range(len(a)):
        j = rng.randrange(len(a))        # BIASED: j from the whole list
        a[i], a[j] = a[j], a[i]

def fisher_yates(a, rng):
    for i in range(len(a) - 1, 0, -1):
        j = rng.randrange(i + 1)         # correct: j only from [0, i]
        a[i], a[j] = a[j], a[i]

def landing_counts(shuffle, trials=3000):
    rng = random.Random(42)              # deterministic
    where = Counter()
    for _ in range(trials):
        a = ["A", "B", "C"]
        shuffle(a, rng)
        where[a.index("B")] += 1         # track B: its naive bias is 10:8:9 out of 27
    return [where[p] for p in range(3)]

print("where 'B' lands over 3000 shuffles of [A, B, C]  (uniform ~ [1000, 1000, 1000])")
print("  naive j in [0, n) :", landing_counts(naive_shuffle), " <- expect ~[1111, 889, 1000]")
print("  Fisher-Yates      :", landing_counts(fisher_yates))

where 'B' lands over 3000 shuffles of [A, B, C]  (uniform ~ [1000, 1000, 1000])
  naive j in [0, n) : [1144, 879, 977]  <- expect ~[1111, 889, 1000]
  Fisher-Yates      : [1006, 995, 999]


## 290 · Backtracking

DFS over partial solutions: make a choice, recurse deeper, then UNDO the choice to try siblings; abandoning doomed branches early is the whole game.

*Theseus in the labyrinth unspooling thread: at each fork he ties a knot (choose), ventures on (explore), and at a dead end rewinds to the knot and unties it (unchoose) to try the next corridor.*

**Watch:** find the choose / explore / unchoose triple in the code. Predict how many 6-queens solutions exist — and what breaks if the `pop()` is deleted.

In [12]:
def solve_n_queens(n):
    solutions = []
    cols, diag1, diag2 = set(), set(), set()
    placement = []                      # placement[row] = chosen column

    def place(row):
        if row == n:
            solutions.append(placement[:])
            return
        for col in range(n):
            if col in cols or (row - col) in diag1 or (row + col) in diag2:
                continue                # doomed branch: prune early
            cols.add(col); diag1.add(row - col); diag2.add(row + col)
            placement.append(col)       # choose
            place(row + 1)              # explore
            placement.pop()             # unchoose
            cols.discard(col); diag1.discard(row - col); diag2.discard(row + col)

    place(0)
    return solutions

sols = solve_n_queens(6)
print("6-queens solutions:", len(sols))
print("first solution:")
for col in sols[0]:
    print("  " + "".join("Q" if c == col else "." for c in range(6)))

6-queens solutions: 4
first solution:
  .Q....
  ...Q..
  .....Q
  Q.....
  ..Q...
  ....Q.
